In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [11]:
df=pd.read_csv('noaa - 5000.csv')
df['date']=pd.to_datetime(df['date'])
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['day'] = df['date'].dt.day

In [ ]:
# Ensure 'date' is the DatetimeIndex
df_time = df.set_index('date').sort_index()

# Extract month and day-of-week attributes
df_time['month_num'] = df_time.index.month             # 1 through 12
df_time['month_name'] = df_time.index.month_name()     # January, February, etc.
df_time['day_of_week'] = df_time.index.day_name()      # Monday, Tuesday, etc.
df_time['day_of_week_num'] = df_time.index.dayofweek  # 0=Monday, 6=Sunday

print(df_time[['temp', 'month_name', 'day_of_week']].head())

monthly_summary = df_time.groupby(['month_num', 'month_name'])['temp'].agg(
    avg_temp='mean',
    min_temp='min',
    max_temp='max',
    count='count'
).reset_index()

print("--- Monthly Seasonal Averages ---")
print(monthly_summary)

weekday_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

dow_summary = df_time.groupby('day_of_week')['temp'].agg(['mean', 'std', 'count']).reindex(weekday_order)

print("\n--- Day-of-Week Averages ---")
print(dow_summary)

# Average temperature by month and day of week
month_dow_pivot = df_time.pivot_table(
    index='month_name',
    columns='day_of_week',
    values='temp',
    aggfunc='mean'
)[weekday_order]

print("\n--- Monthly vs Day-of-Week Matrix ---")
print(month_dow_pivot)


fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Plot 1: Monthly seasonal trajectory
axes[0].plot(monthly_summary['month_name'], monthly_summary['avg_temp'], marker='o', color='crimson', linewidth=2)
axes[0].fill_between(
    monthly_summary['month_name'],
    monthly_summary['min_temp'],
    monthly_summary['max_temp'],
    alpha=0.15,
    color='crimson',
    label='Min-Max Range'
)
axes[0].set_title('Seasonal Temperature Pattern by Month', fontweight='bold')
axes[0].set_xlabel('Month')
axes[0].set_ylabel('Average Temperature (°F)')
axes[0].tick_params(axis='x', rotation=45)
axes[0].grid(True, linestyle='--', alpha=0.5)
axes[0].legend()

# Plot 2: Day-of-week consistency
axes[1].bar(dow_summary.index, dow_summary['mean'], color='cornflowerblue', edgecolor='black', alpha=0.85)
axes[1].set_title('Average Temperature by Day of Week', fontweight='bold')
axes[1].set_xlabel('Day of Week')
axes[1].set_ylabel('Average Temperature (°F)')
axes[1].tick_params(axis='x', rotation=45)
axes[1].grid(True, linestyle='--', alpha=0.5, axis='y')

plt.tight_layout()
plt.show()


In [ ]:


# 1. Compute Daily Mean Temperature (sorted chronologically)
daily_temp = df.groupby('date')['temp'].mean().sort_index()

# 2. Compute 30-Day Rolling Mean (smooths out daily spikes)
rolling_30d = daily_temp.rolling(window=30, min_periods=1).mean()

# 3. Compute Monthly Resampled Averages ('ME' for Month-End in modern pandas)
monthly_temp = daily_temp.resample('ME').mean()

# ----------------- PLOTTING -----------------
plt.figure(figsize=(14, 6))

# Layer 1: Raw Daily Readings (faded to show variance without cluttering)
plt.plot(
    daily_temp.index, 
    daily_temp.values, 
    label='Daily Average Temp (Raw)', 
    color='gray', 
    alpha=0.4, 
    linewidth=1
)

# Layer 2: Smoothed 30-Day Rolling Trend (bold continuous curve)
plt.plot(
    rolling_30d.index, 
    rolling_30d.values, 
    label='30-Day Rolling Average', 
    color='#E63946', 
    linewidth=2.5
)

# Layer 3: Monthly Resampled Points (prominent markers with dashed guide line)
plt.plot(
    monthly_temp.index, 
    monthly_temp.values, 
    label="Monthly Resampled Average ('ME')", 
    color='#1D3557', 
    marker='o', 
    markersize=8, 
    linestyle='--', 
    linewidth=1.8
)

# Formatting
plt.title('Daily Weather Observations, 30-Day Rolling Trend & Monthly Averages', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Date', fontsize=12)
plt.ylabel('Temperature (°F)', fontsize=12)
plt.legend(frameon=True, facecolor='white', framealpha=0.9, loc='upper left', fontsize=10)
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()

plt.show()

# Add this right before plotting Layer 1:
plt.scatter(
    df['date'], 
    df['temp'], 
    color='steelblue', 
    alpha=0.12, 
    s=12, 
    label='Raw Station Observations'
)


In [12]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 35 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   stn                   5000 non-null   int64         
 1   wban                  5000 non-null   int64         
 2   date                  5000 non-null   datetime64[us]
 3   year                  5000 non-null   int32         
 4   mo                    5000 non-null   int64         
 5   da                    5000 non-null   int64         
 6   temp                  5000 non-null   float64       
 7   count_temp            5000 non-null   int64         
 8   dewp                  5000 non-null   float64       
 9   count_dewp            5000 non-null   int64         
 10  slp                   5000 non-null   float64       
 11  count_slp             5000 non-null   int64         
 12  stp                   5000 non-null   float64       
 13  count_stp             5000 no